# 第三阶段（一）：业务特征工程

本 Notebook 只负责教学展示，正式公式位于 `src.feature_engineering.features`。输入为第二阶段清洗数据；目标变量不参与特征计算。

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.feature_engineering.features import (
    CANDIDATE_FEATURES, DERIVED_FEATURES, FEATURE_DEFINITIONS, FeatureEngineer
)

## 特征公式

所有收入和逾期衍生变量都会利用第二阶段质量标记恢复原始缺失状态，避免把填充值误当成真实业务数据。

In [2]:
pd.DataFrame([{'feature': key, '定义': value} for key, value in FEATURE_DEFINITIONS.items()])

,feature,定义
0,total_delinquency_count,30–59天、60–89天和90天及以上历史逾期次数之和
1,severe_delinquency_flag,60–89天或90天及以上历史逾期次数大于0记为1
2,delinquency_severity_score,30–59天次数 + 2×60–89天次数 + 3×90天及以上次数
3,real_estate_loan_share,房地产贷款或额度数 ÷ 开放信贷账户数
4,monthly_income_per_capita,月收入 ÷（家属人数 + 1）
5,estimated_monthly_debt,月收入 × 债务比率
6,residual_income_per_capita,月收入 ×（1−债务比率）÷（家属人数 + 1）


In [3]:
train_clean = pd.read_csv(ROOT / 'data' / 'processed' / 'train_clean.csv')
train_features = FeatureEngineer().transform(train_clean)
{
    '训练样本数': len(train_features),
    '原始变量数': 10,
    '衍生变量数': len(DERIVED_FEATURES),
    '正式候选变量数': len(CANDIDATE_FEATURES),
    '目标变量进入特征计算': 'SeriousDlqin2yrs' in train_features.columns,
}

{'训练样本数': 105000, '原始变量数': 10, '衍生变量数': 7, '正式候选变量数': 20, '目标变量进入特征计算': False}

## 衍生变量缺失与分布检查

这里的缺失是有意保留给 OptimalBinning 的 Missing 箱，不再进行二次中位数填补。

In [4]:
pd.DataFrame({
    '非缺失数': train_features[list(DERIVED_FEATURES)].notna().sum(),
    '缺失数': train_features[list(DERIVED_FEATURES)].isna().sum(),
    '缺失率': train_features[list(DERIVED_FEATURES)].isna().mean(),
    '中位数': train_features[list(DERIVED_FEATURES)].median(),
}).sort_values('缺失率', ascending=False)

,非缺失数,缺失数,缺失率,中位数
residual_income_per_capita,83131,21869,0.208276,2221.314891
estimated_monthly_debt,83131,21869,0.208276,1653.745581
monthly_income_per_capita,83131,21869,0.208276,3383.500000
real_estate_loan_share,103679,1321,0.012581,0.100000
delinquency_severity_score,104818,182,0.001733,0.000000
total_delinquency_count,104818,182,0.001733,0.000000
severe_delinquency_flag,104818,182,0.001733,0.000000
